In [2]:
#pip install undetected_chromedriver

  Preparing metadata (setup.py) ... done
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 17.8 MB/s eta 0:00:00a 0:00:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
  DEPRECATION: Building 'undetected_chromedriver' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'undetected_chromedriver'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for undetected_chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47048 sha256=dab342d93

In [28]:
'''import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time

Model = ['Hyundai', 'Kia']
data = []

# ---> PUT YOUR ZENROWS API KEY HERE! <---
ZENROWS_API_KEY = "b3ef00babc291e3e6032e4ee03ed02aaabcfa424"

for i in Model:
    url = "https://partsouq.com/en/catalog/genuine/pick?c=" + i + "&model=Passenger&ssd=%24%2AKwHr18fImIqbmJaKkovBwseci-2egqP9s-Hj9_vi_ZOG6f3xovjzta26_OP67f_90wAAAACxm2NA%24"
    
    # Configure the powerful ZenRows API completely masking our identity
    zenrows_endpoint = "https://api.zenrows.com/v1/"
    params = {
        "apikey": ZENROWS_API_KEY,
        "url": url, 
        "antibot": "true",
        "premium_proxy": "true", # Demolishes Cloudflare Turnstile
        "js_render": "true",               # Instructs ZenRows to fire up a hidden headless browser and run JS
        "wait_for": ".search-result-vin"   # Tells ZenRows to STARE at the screen and WAIT for this table to visually appear!
    }
    
    # ── Retry up to 3 times ──
    success = False
    for attempt in range(3):
        try:
            print(f"⏳ {i} — attempt {attempt+1}, asking ZenRows to bypass Cloudflare and wait for the table...")
            
            # Since ZenRows is manually defeating the Turnstile, waiting for page load, AND waiting for the table, it takes time. 
            response = requests.get(zenrows_endpoint, params=params, timeout=60)
            
            if response.status_code == 200:
                print("✅ Turnstile bypassed fully! Searching the massive HTML for the table...")
                
                # We hand the massive HTML to BeautifulSoup to instantly hunt down our table
                soup = BeautifulSoup(response.content, "html.parser")
                table_div = soup.find(class_="search-result-vin")
                
                if table_div:
                    rows = table_div.find_all("tr")
                    
                    for row in rows:
                        cells = row.find_all("td")
                        
                        if cells:
                            row_data = [cell.text.strip() for cell in cells]
                            
                            # Safely extract href link like you did before
                            a_tag = cells[0].find("a")
                            href = a_tag["href"] if a_tag else ""
                            
                            row_data.append(href)
                            row_data.append(i)
                            data.append(row_data)
                    
                    print(f"✅ {i} done — {len(data)} rows so far")
                    success = True
                    break  # exit retry loop if successful
                else:
                    # Very strange, ZenRows swore the table was there but bs4 didn't see it (this almost never happens with wait_for).
                    print("⚠️ The page loaded, but the table 'search-result-vin' was totally missing.")
                    print("Could be intermittent Cloudflare block, sleeping 10s and trying again...")
                    time.sleep(10)
            else:
                # ZenRows officially threw a visible error
                print(f"⚠️ ZenRows failed with Code {response.status_code}: {response.text}")
                time.sleep(10)  # wait before retrying

        except Exception as e:
            print(f"⚠️ Attempt {attempt+1} absolutely failed for {i}: {e}")
            time.sleep(10)  # wait before retrying
    
    if not success:
        print(f"❌ Skipping {i} — all 3 attempts completely failed to grab the table.")

# Clean up and shape the final DataFrame exactly how you designed it!
vehicle = pd.DataFrame(data, columns=["Model", "Market", "Year From", "Year To", "Notes", 'URL', 'Make'])
vehicle['URL2'] = np.where(vehicle['Make'].isna(), vehicle['Notes'], vehicle['URL'])
vehicle['Make'] = np.where(vehicle['Make'].isna(), vehicle['URL'], vehicle['Make'])
vehicle['URL'] = vehicle['URL2']
vehicle.drop(['Notes', 'URL2'], inplace=True, axis=1)
vehicle['URL'] = vehicle['URL'].str.replace('vehicle', 'groups')

# ---> SAVE THE RESULTS AS A CSV INTO THE FOLDER <---
vehicle.to_csv('output/Scraped_Vehicles.csv', index=False)
'''

'import pandas as pd\nimport numpy as np\nimport requests\nfrom bs4 import BeautifulSoup\nimport time\n\nModel = [\'Hyundai\', \'Kia\']\ndata = []\n\n# ---> PUT YOUR ZENROWS API KEY HERE! <---\nZENROWS_API_KEY = "b3ef00babc291e3e6032e4ee03ed02aaabcfa424"\n\nfor i in Model:\n    url = "https://partsouq.com/en/catalog/genuine/pick?c=" + i + "&model=Passenger&ssd=%24%2AKwHr18fImIqbmJaKkovBwseci-2egqP9s-Hj9_vi_ZOG6f3xovjzta26_OP67f_90wAAAACxm2NA%24"\n    \n    # Configure the powerful ZenRows API completely masking our identity\n    zenrows_endpoint = "https://api.zenrows.com/v1/"\n    params = {\n        "apikey": ZENROWS_API_KEY,\n        "url": url, \n        "antibot": "true",\n        "premium_proxy": "true", # Demolishes Cloudflare Turnstile\n        "js_render": "true",               # Instructs ZenRows to fire up a hidden headless browser and run JS\n        "wait_for": ".search-result-vin"   # Tells ZenRows to STARE at the screen and WAIT for this table to visually appear!\n    }\

In [20]:
#vehicle['URL']='https://partsouq.com'+vehicle['URL']

In [ ]:
'''vehicle.to_excel("output/vehicle.xlsx", index=False)'''

In [ ]:
'''test'''

In [27]:
import concurrent.futures
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
import os

vehicle=pd.read_csv('output/Scraped_Vehicles.csv')

vehicle['URL']='https://partsouq.com'+vehicle['URL']
# ---> PUT YOUR ZENROWS API KEY HERE! <---
ZENROWS_API_KEY = "b3ef00babc291e3e6032e4ee03ed02aaabcfa424"

# The cold-storage file where we keep our safe partial records
RAW_CHECKPOINT_FILE = 'output/Category_URL_Raw.csv' #'output/'

def fetch_and_parse_categories(idx, target_url):
    """Hits an individual vehicle URL and extracts all treegrid links asynchronously without clicking"""
    
    # Fix: Reconstruct absolute URL if BeautifulSoup grabbed a relative one in Phase 1
       
    zenrows_endpoint = "https://api.zenrows.com/v1/"
    params = {
        "apikey": ZENROWS_API_KEY,  
        "url": target_url, 
        "antibot": "true",
        "premium_proxy": "true",
        "js_render": "true" 
    }
    
    # Retry loop strictly for this one specific thread
    for attempt in range(3):
        try:
            # We timeout at 120s to ensure ZenRows queueing and JS-rendering completes fully
            response = requests.get(zenrows_endpoint, params=params, timeout=120)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, "html.parser")
                treegrids = soup.find_all("tr", class_=lambda c: c and "treegrid" in c)
                
                extracted_links = []
                for tr in treegrids:
                    a_tag = tr.find("a")
                    if a_tag and a_tag.get("href"):
                        text = a_tag.text.strip()
                        href = a_tag["href"]
                        extracted_links.append((idx, text, href))
                
                print(f"✅ Finished Vehicle Row [{idx}] — Extracted {len(extracted_links)} deep links.")
                return extracted_links
            else:
                print(f"⚠️ ZENROWS ERROR on Row [{idx}]: Code {response.status_code} | {response.text}")
                time.sleep(5)
                
        except Exception as e:
            print(f"⚠️ PYTHON ERROR on Row [{idx}]: {e}")
            time.sleep(5)
            
    print(f"❌ Failed all 3 attempts to grab Vehicle Row [{idx}]. Handing back empty array to skip.")
    return []


# =========================================================
# THE SMART RESUME ENGINE
# =========================================================

scraped_rows = set()

# 1. Check if we crashed recently and have progress to load
if os.path.exists(RAW_CHECKPOINT_FILE):
    existing_df = pd.read_csv(RAW_CHECKPOINT_FILE)
    if not existing_df.empty and 'rownumber' in existing_df.columns:
        scraped_rows = set(existing_df['rownumber'].unique())
        print(f"🔄 Checkpoint found! Skipping {len(scraped_rows)} vehicle groups we already scraped earlier!")
else:
    # We are starting fresh! Create the empty CSV file with correct headers.
    pd.DataFrame(columns=['rownumber', 'SubCategory', 'SubCat_URL']).to_csv(RAW_CHECKPOINT_FILE, index=False)
    print("✨ Starting completely fresh scrape. Created empty checkpoint file.")


# 2. Filter down our massive list to ONLY the cars we haven't scraped yet
targets_remaining = [(i, url) for i, url in enumerate(vehicle['URL'][:5]) if i not in scraped_rows]
print(f"🚀 Igniting ThreadPoolExecutor for the remaining {len(targets_remaining)} vehicles...")

# ---> DROPPED TO 2 WORKERS TO PREVENT ZENROWS 1000-CREDIT BURN & 429 ERRORS <---
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    
    # Throw the un-scraped targets into the thread pool
    futures = {executor.submit(fetch_and_parse_categories, idx, url): idx 
               for idx, url in targets_remaining}
    
    for future in concurrent.futures.as_completed(futures):
        idx = futures[future]
        try:
            res = future.result()
            if res:
                # The microsecond the thread finishes, the MAIN program thread instantly packs its data
                # into a mini dataframe and appends it to the cold storage CSV file on your hard drive!
                temp_df = pd.DataFrame(res, columns=['rownumber', 'SubCategory', 'SubCat_URL'])
                temp_df.to_csv(RAW_CHECKPOINT_FILE, mode='a', header=False, index=False)
        except Exception as e:
            print(f"💥 Complete critical failure processing row {idx}: {e}")

print(f"🏁 Threading Complete! All vehicles processed.")

# =========================================================
# FINAL MERGE AND CLEANUP
# =========================================================

# Now that all threads are done, read out the completely stuffed checkpoint file
final_raw_df = pd.read_csv(RAW_CHECKPOINT_FILE)

# Merge logic utterly unchanged from your original script!


print(f"🎉 Merged securely! Total catalog DataFrame rows: {len(final_raw_df)}")

# Save the final beautiful output!
final_raw_df.to_csv('output/Category_URL_Final.csv', index=False)
print("💾 ALL DONE! Saved to output/Category_URL_Final.csv")

🔄 Checkpoint found! Skipping 2 vehicle groups we already scraped earlier!
🚀 Igniting ThreadPoolExecutor for the remaining 3 vehicles...
✅ Finished Vehicle Row [2] — Extracted 409 deep links.
✅ Finished Vehicle Row [3] — Extracted 434 deep links.
✅ Finished Vehicle Row [4] — Extracted 250 deep links.
🏁 Threading Complete! All vehicles processed.
🎉 Merged securely! Total catalog DataFrame rows: 496
💾 ALL DONE! Saved to output/Category_URL_Final.csv


,rownumber,SubCategory,SubCat_URL
0,0,Oil Filter,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
1,0,Air Filter,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
2,0,Fuel Filter,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
3,0,"Air Filter, passenger compartment",/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
4,0,Spark Plug,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
...,...,...,...
1965,4,Air Bag System,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
1966,4,Air Bag System,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
1967,4,Safety Belt System,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...
1968,4,Armrest,/en/catalog/genuine/parts?c=Hyundai&ssd=%24%2A...


In [ ]:
'''part_data = []
for i in range(len(Category_URL['SubCat_URL'])):
    driver.get(Category_URL['SubCat_URL'][i])
   

    # Get category
    category = driver.find_element(
        By.XPATH, "//div[contains(@class,'col-lg-12')]//h2[@style='color: indigo;']"
    ).text.strip().replace("\n", " ")
    
    # Get all subcategory blocks
    blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'unit-header')]/ancestor::div[@class='row']")
    
    for block in blocks:
        # Subcategory name
        try:
            subcategory = block.find_element(
                By.XPATH, ".//div[contains(@class,'unit-header')]//h2"
            ).text.strip()
        except:
            subcategory = ""
    
        # Image URL
        try:
            img = block.find_element(By.XPATH, ".//img[contains(@class,'drag')]")
            img_url = img.get_attribute("src")
        except:
            img_url = ""
    
        # Rows inside this block
        rows = block.find_elements(By.CLASS_NAME, "part-search-tr")
    
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "td")
            if len(cells) >= 5:
                try:
                    number_link = cells[0].find_element(By.TAG_NAME, "a")
                    number      = number_link.text.strip()
                    number_url  = number_link.get_attribute("href")
                except:
                    number      = cells[0].text.strip()
                    number_url  = ""
    
                part_data.append({
                    "category":    category,
                    "subcategory": subcategory,
                    "img_url":     img_url,
                    "number":      number,
                    "number_url":  number_url,
                    "name":        cells[1].text.strip(),
                    "code":        cells[2].text.strip(),
                    "date_range":  cells[3].text.strip(),
                    "options":     cells[4].text.strip(),
                    "quantity":    cells[5].text.strip() if len(cells) > 5 else "",
                    'rownumber':   i
                })



part_data = []

# Get category
category = driver.find_element(
    By.XPATH, "//div[contains(@class,'col-lg-12')]//h2[@style='color: indigo;']"
).text.strip().replace("\n", " ")

# Get all subcategory blocks
blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'unit-header')]/ancestor::div[@class='row']")

for block in blocks:
    # Subcategory name
    try:
        subcategory = block.find_element(
            By.XPATH, ".//div[contains(@class,'unit-header')]//h2"
        ).text.strip()
    except:
        subcategory = ""

    # Image URL
    try:
        img = block.find_element(By.XPATH, ".//img[contains(@class,'drag')]")
        img_url = img.get_attribute("src")
    except:
        img_url = ""

    # Rows inside this block
    rows = block.find_elements(By.CLASS_NAME, "part-search-tr")

    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        if len(cells) >= 5:
            try:
                number_link = cells[0].find_element(By.TAG_NAME, "a")
                number      = number_link.text.strip()
                number_url  = number_link.get_attribute("href")
            except:
                number      = cells[0].text.strip()
                number_url  = ""

            part_data.append({
                "category":    category,
                "subcategory": subcategory,
                "img_url":     img_url,
                "number":      number,
                "number_url":  number_url,
                "name":        cells[1].text.strip(),
                "code":        cells[2].text.strip(),
                "date_range":  cells[3].text.strip(),
                "options":     cells[4].text.strip(),
                "quantity":    cells[5].text.strip() if len(cells) > 5 else ""
            })
Final_data=Category_URL.merge(pd.DataFrame(part_data),left_index=True,right_on='rownumber')
Final_data=Final_data[['Make','Model', 'Market', 'Year From', 'Year To', 'URL',
            'SubCategory', 'SubCat_URL', 'category', 'subcategory',
            'img_url', 'number', 'number_url', 'name', 'code', 'date_range',
            'options', 'quantity']]
'''

In [ ]:
'''driver.get('https://www.autodoc.co.uk/nty/213502B703')

all_data = []

for i in range(1):
    oem = str(Final_data['number'][i])

    driver.get(f"https://www.autodoc.co.uk/car-parts/oem/{oem}")
    time.sleep(3)

    try:
        first_product = driver.find_element(By.CLASS_NAME, "listing-item__name")
        first_product.click()
        time.sleep(3)
    except:
        print(f"No result for {oem}")
        continue

    data = {}
    data["OEM"] = oem

    # Info table
    for item in driver.find_elements(By.CLASS_NAME, "product-description__item"):
        try:
            label = item.find_element(By.CLASS_NAME, "product-description__item-title").text.strip()
            value = item.find_element(By.CLASS_NAME, "product-description__item-value").text.strip()
            data[label] = value
        except:
            pass

    # Images
    img_urls = []
    for item in driver.find_elements(By.CLASS_NAME, "product-gallery__image-list-item"):
        try:
            url = item.find_element(By.TAG_NAME, "img").get_attribute("src")
            if url:
                img_urls.append(url)
        except:
            pass
    data["images"] = img_urls

    # Scroll gradually to trigger lazy loading
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.3);")
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.6);")
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.9);")
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.6);")
    time.sleep(2)

    # Compatibility
    compat = []
    make_blocks = driver.find_elements(By.CLASS_NAME, "product-info-block__item")
    print(f"  Make blocks found: {len(make_blocks)}")

    for make_block in make_blocks:
        try:
            make = make_block.find_element(By.CLASS_NAME, "product-info-block__item-title").get_attribute("innerHTML").strip()
        except:
            make = ""

        driver.execute_script("arguments[0].click();", make_block.find_element(By.CLASS_NAME, "product-info-block__item-title"))
        time.sleep(1)

        model_items = make_block.find_elements(By.CLASS_NAME, "product-info-block__item-list__item")
        for model_item in model_items:
            try:
                model = model_item.find_element(By.CLASS_NAME, "product-info-block__item-list__title").get_attribute("innerHTML").strip()
            except:
                model = ""

            driver.execute_script("arguments[0].click();", model_item.find_element(By.CLASS_NAME, "product-info-block__item-list__title"))
            time.sleep(0.5)

            engines = model_item.find_elements(By.XPATH, ".//ul[contains(@class,'product-info-block__item-sublist')]//li")
            engine_list = [e.get_attribute("innerHTML").strip() for e in engines if e.get_attribute("innerHTML").strip()]

            if engine_list:
                for engine in engine_list:
                    compat.append({"make": make, "model": model, "engine": engine})
            else:
                compat.append({"make": make, "model": model, "engine": ""})

    data["compatibility"] = str(compat)
    all_data.append(data)
    print(f"✅ {oem} done — {len(compat)} compat rows")

autodoc_data_overall = pd.DataFrame(all_data)

autodoc_data_overall=pd.DataFrame([data])



# Parse the string back to list of dicts
autodoc_data_overall['compatibility'] = autodoc_data_overall['compatibility'].apply(ast.literal_eval)

# Explode compatibility into separate rows
df_exploded = autodoc_data_overall.explode('compatibility').reset_index(drop=True)

# Split compatibility dict into separate columns
df_exploded = pd.concat([
    df_exploded.drop(columns=['compatibility']),
    df_exploded['compatibility'].apply(pd.Series)
], axis=1)

Final_data2=Final_data.merge(df_exploded,left_on='number',right_on='OEM')
'''